# 02 · 데이터 이해하기

## 1. 파일 위치 찾기

Path.cwd()는 현재 폴더입니다. / 기호는 여기서는 경로 조각을 잇습니다. 문자열 안의 /와 숫자 나눗셈은 쓰이는 문맥이 다릅니다.

## 0. 실습 폴더 준비

VS Code와 Colab 어디서 열어도 `data` 폴더를 찾도록 실습 폴더 위치를 정합니다. Colab에서는 처음 한 번 실습 자료를 내려받습니다.

In [ ]:
import sys, subprocess  # 실행 환경 확인과 명령 실행 도구입니다.
from pathlib import Path  # 파일 위치 도구입니다.
if "google.colab" in sys.modules:  # Colab에서 실행 중이라면
    folder = Path("/content/semiconductor_ai_beginner")  # 실습 자료를 둘 위치입니다.
    if not folder.exists():  # 아직 내려받지 않았다면
        subprocess.run(["git", "clone", "-q", "https://github.com/SeokJH-TAs/semiconductor_ai_beginner.git", str(folder)], check=True)  # 실습 자료 전체를 내려받습니다.
else:  # VS Code 등 내 컴퓨터에서 실행 중이라면
    folder = Path.cwd()  # 현재 폴더입니다.
    if folder.name == "notebooks":  # 노트북 폴더 안이라면
        folder = folder.parent  # 실습 전체 폴더로 올라갑니다.
print(folder)  # 실습 폴더 위치를 확인합니다.

In [ ]:
from pathlib import Path  # 파일 위치 도구입니다.
import pandas as pd  # 표를 읽는 도구입니다.
file_path = folder / "data" / "process.csv"  # 폴더와 파일 이름을 연결합니다.
print(file_path)  # 읽을 파일 위치를 봅니다.

## 2. 표 열고 크기 읽기

CSV는 행과 열을 쉼표로 구분한 파일입니다. read_csv는 파일을 읽어 표로 만듭니다.

In [2]:
data = pd.read_csv(file_path)  # CSV를 표로 읽습니다.
print(data.head())  # 처음 다섯 행을 봅니다.
print(data.shape)  # 240행과 7열인지 확인합니다.
print(data.columns.tolist())  # 열 이름을 목록으로 봅니다.

  sample_id  temperature_c  pressure_torr  gas_flow_sccm  rf_power_w  \
0      W001        413.698          1.664        110.439     221.019   
1      W002        396.944          1.637        100.881     214.638   
2      W003        417.930          1.186        119.415     182.867   
3      W004        409.868          1.425        105.852     256.782   
4      W005        379.709          1.485         91.803     188.240   

   thickness_nm  defect  
0       104.018       1  
1        97.325       0  
2       107.485       1  
3       100.399       0  
4        92.937       0  
(240, 7)
['sample_id', 'temperature_c', 'pressure_torr', 'gas_flow_sccm', 'rf_power_w', 'thickness_nm', 'defect']


## 3. 단위와 정답 비율 확인

0/1 라벨의 평균은 1인 행의 비율입니다. 여기서 얻은 값은 웨이퍼 기록의 불량 라벨 비율이며 칩 수율이 아닙니다.

In [3]:
print(data["temperature_c"].min())  # 표에서 가장 낮은 공정 온도입니다.
print(data["temperature_c"].max())  # 가장 높은 공정 온도입니다.
print(data["defect"].value_counts())  # 정상·불량 라벨 개수를 셉니다.
print(data["defect"].mean())  # 불량 라벨을 가진 행의 비율입니다.

375.368
424.619
defect
0    178
1     62
Name: count, dtype: int64
0.25833333333333336


## 4. 데이터의 네 모양

방금 읽은 표는 정형 데이터입니다. 시계열은 순서가 정보입니다. 이미지는 밝기 격자, 텍스트는 글자의 흐름입니다. 아래 작은 예는 모양 관찰용이고 이미지·언어 모델 훈련은 아닙니다.

In [4]:
import numpy as np  # 숫자 격자 도구입니다.
time_data = pd.read_csv(folder / "data" / "temperature_log.csv")  # 시간 순서대로 쌓인 기록입니다.
image = np.array([[0, 0, 0], [0, 255, 0], [0, 0, 0]])  # 가운데만 밝은 3×3 흑백 격자입니다.
text = "챔버 온도를 확인해 주세요"  # 글자 데이터 예입니다.
print(time_data.head())  # 분 단위 순서가 있는 표를 봅니다.
print(image)  # 이미지도 숫자 격자로 표현됨을 봅니다.
print(text)  # 텍스트 예입니다.

   minute  temperature_c
0       0            399
1       1            400
2       2            401
3       3            400
4       4            400
[[  0   0   0]
 [  0 255   0]
 [  0   0   0]]
챔버 온도를 확인해 주세요


# 데이터 안내

모든 CSV는 이 수업을 위해 만든 **교육용 합성 데이터**입니다. 실제 기업·공장·제품의 측정 자료가 아니며 원본 PDF에 실린 산업 데이터도 아닙니다. NumPy 난수 시드 42를 시작점으로 생성했고, 데이터 생성의 설계식은 아래에 공개했습니다. 작은 표의 경향을 배우는 목적입니다.

## 파일별 한 행의 뜻

| 파일 | 행×열 | 한 행의 뜻 | 사용 실습 |
|---|---:|---|---|
| process.csv | 240×7 | 서로 독립으로 만든 가상 웨이퍼 한 장의 공정 요약 | 0103, 0303, 0403, 0603 |
| process_dirty.csv | 13×7 | process 앞 12행에 빈칸 2개·중복 1행 삽입 | 0201 |
| clusters.csv | 90×2 | 세 가지 가상 공정 조건 중 한 기록 | 0202 |
| sensors.csv | 120×3 | 함께 움직이는 센서 세 개의 참조 기록 | 0203 |
| sensor_check.csv | 2×3 | 정상 유사 기록 한 개와 센서 하나만 튀는 기록 한 개 | 0203 |
| temperature_log.csv | 12×2 | 1분마다 한 번 측정한 온도 예시 | 데이터 안내 |

## process.csv 열 설명

| 열 | 쉬운 뜻 | 단위·범위 | 예측 시점 | 사용 역할 |
|---|---|---|---|---|
| sample_id | 웨이퍼 구분 이름 | W001–W240 | 식별자 | 입력에서 제외 |
| temperature_c | 공정 온도 | °C, 생성 범위 375–425 | 공정 중 조건 | 입력 후보 |
| pressure_torr | 공정 압력 | Torr, 1.0–1.8 | 공정 중 조건 | 입력 후보. 생성식에서는 결과와 무관한 잡음 특징 |
| gas_flow_sccm | 가스 유량 | sccm, 90–140 | 공정 중 조건 | 입력 후보 |
| rf_power_w | RF 전력 | W, 180–260 | 공정 중 조건 | 입력 후보 |
| thickness_nm | 공정 후 막 두께 | nm, 100 근처 | 공정 후 측정 | 두께 회귀의 정답; 사전 불량 분류 입력 제외 |
| defect | 공정 후 검사 결과 | 0=정상, 1=불량 | 공정 후 검사 | 불량 분류의 정답 |

단위는 예제 해석을 위한 표기입니다. 수치와 관계는 실제 레시피·공정 허용 범위를 뜻하지 않습니다. 한 웨이퍼에 라벨 하나를 붙인 간단한 예이며 칩별 불량 맵이나 칩 수율 기록이 아닙니다.

## 데이터 생성 원리

공정 조건 네 개는 각각 독립 균등분포로 만들었습니다. 두께는 다음 식에 평균 0, 표준편차 1.2nm인 정규 잡음을 더했습니다.

`두께 = 100 + 0.3×(온도−400) + 0.12×(유량−115) − 0.04×(전력−220) + 잡음`

가상 위험 점수는 `abs(두께−100)/3 + abs(온도−400)/18 + 잡음`입니다. 여기서 잡음은 평균 0·표준편차 0.5입니다. 점수가 2.9보다 크면 defect=1입니다. 저장 숫자는 소수 셋째 자리로 반올림했습니다. 이 라벨 규칙은 교육용이며 현장의 불량 판정 기준이 아닙니다. 이 데이터에서 높은 점수를 얻어도 실제 공정에서 사용할 수 있다는 뜻은 아닙니다.

clusters는 중심 (385,1.1), (400,1.5), (417,1.15) 근처에서 각 30개를 만들었습니다. 온도 잡음 표준편차는 2, 압력은 0.035입니다. 그룹 정답은 파일에 저장하지 않았습니다.

sensors는 같은 잠재 숫자 t를 공유합니다. a=100+3t+잡음, b=50+2t+잡음, c=20−t+잡음입니다. t의 평균은 0·표준편차는 1, 센서 잡음의 표준편차는 각각 0.3,0.3,0.15입니다. check의 첫 행은 (100,50,20), 둘째 행은 (130,50,20)입니다. 둘째는 a만 30 크게 만든 것입니다. 분류 성능을 통계적으로 평가하는 시험 집합은 아닙니다.

## 입력 선택과 누출

무엇을 예측하기 **전**에 알 수 있는 열인지 먼저 확인합니다. 두께를 예측하면서 thickness_nm을 입력으로 넣으면 정답을 미리 보는 것입니다. 사전 불량 예측에 공정 후에만 얻는 두께를 넣으면 사용 시점과 맞지 않습니다. sample_id의 번호에도 의미 있는 공정 지식이 없습니다.

## 나누기 원칙

0303·0403은 학습 75% / 시험 25%로 고정 모델의 성능을 확인합니다. 0603은 학습 60% / 검증 20% / 최종 시험 20%로 설정을 선택합니다. StandardScaler는 학습 데이터로만 기준을 배우고 새 데이터에는 transform만 적용합니다. 0202는 표 전체를 탐색하는 비지도 실습, 0203은 참조 표로만 압축 기준을 배우는 새 기록 확인 실습입니다.

이 파일은 독립 합성 행이라 무작위 분할을 씁니다. 실제 공정에서 같은 로트·웨이퍼의 여러 행을 나누거나 미래가 과거 학습에 들어가면 누출이 생길 수 있습니다. 그런 자료는 로트·장비·시간 기준으로 나누는 설계가 필요합니다.

## 직접 확인하기

온도 단위는 무엇인가요? 두께 예측 때 정답 열은 무엇인가요? 불량을 미리 예측할 때 공정 후 두께를 입력으로 쓰면 왜 문제가 될까요?

**해설:** °C, thickness_nm입니다. 예측할 때 아직 알 수 없는 정보를 쓰게 되므로 실제 사용 조건과 다릅니다.